# Track 1 · Step 8: test evaluation (once)

AI251P Capstone, Phase 1 (M2: per-class P/R/F1, confusion matrix, calibration, error gallery; M1: CI for test accuracy). Owner: Abdulmohsen Alharthi.

The model was chosen on val (Steps 5–7): **cls-full-64, lr 0.002** (LR check on val, 23 Oct: 3-seed mean 0.817 against 0.802 at lr 0.001). This notebook opens the test split once and evaluates, with nothing tuned on test:

1. **Checkpoints**: the 3 seeds of the selected model, plus the 3 seeds of scratch and frozen at the same size for the M2–M3 comparison (remove them from `EVAL` if you only want the selected model). The logit bias stored in each checkpoint (fitted on val) is applied unchanged.
2. **Per run, raw and logit-adjusted**: accuracy, macro-F1, macro-F1 per domain (sea / urban), per-class precision / recall / F1, ECE (15 bins). Mean ± std over seeds.
3. **95 % confidence intervals**: bootstrap over **groups** (SeaDronesSee videos and UAVDT sequences, resampled within each dataset, 2,000 draws, seeded), because crops from one video or sequence are not independent; a crop-level Wilson interval for accuracy is shown too, for reference only. The metric in each draw is the mean over the 3 seeds.
4. **Group counts per class** in test (crops, videos/sequences, tracks), next to every per-class number.
5. **Confusion matrices** (3 seeds pooled, raw and adjusted), **reliability diagram**, **recall by object size**, and an **error gallery** of the most frequent confusions.
6. **Test log**: every opening of test is appended to `02_Classification/TestLog.csv` with the checkpoints used, so the report can state how many times test was looked at.

Outputs: `02_Classification/…_TestResults_vN.json`, `…_TestPerClass_vN.csv`, `…_TestConfusion_<model>_vN.png/.csv`, `…_TestReliability_vN.png`, `…_TestRecallBySize_vN.csv`, `…_ErrorGallery_vN.png`, `…_TestErrors_vN.csv.gz`, `…_TestPredictions_vN.csv.gz`.

Runtime: GPU or CPU (inference only, a few minutes).

## 0. Settings, mount Drive

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
SEEDS = [42, 43, 44]
SELECTED = 'full FT'
EVAL = {'full FT': 'cls-full-64-s{seed}-lr0.002',   # selected on val (23 Oct LR check: lr 0.002 beat 0.001 on all 3 seeds)
        'full FT lr0.001': 'cls-full-64-s{seed}',     # previous default lr, for the LR comparison on test
        'scratch': 'cls-scratch-64-s{seed}',
        'frozen':  'cls-frozen-64-s{seed}'}
N_BOOT, SEED = 2000, 42
GALLERY_PAIRS, GALLERY_PER_PAIR = 4, 8              # most frequent (true -> predicted) pairs, crops per pair

!pip -q install pyyaml
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile /content/aerial_crops.py
"""Aerial object crops for the AI251P 8-class classifier (Track 1).

Crops are stored by Step 4 as one uint8 array per split, shape (N, S, S, 3), plus an index CSV whose
`row` column points into the array. Labels follow CLASSES below.
"""
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T

CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, used for scratch and pretrained alike


class Rot90:
    """Random multiple of 90 degrees: aerial views have no fixed 'up'."""
    def __call__(self, x):
        return torch.rot90(x, int(torch.randint(4, (1,))), dims=(-2, -1))


def make_transforms(img_size, train):
    if train:
        return T.Compose([T.ToImage(), T.RandomResizedCrop(img_size, scale=(0.75, 1.0), ratio=(0.9, 1.1), antialias=True),
                          T.RandomHorizontalFlip(), Rot90(),
                          T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
                          T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    return T.Compose([T.ToImage(), T.Resize(img_size, antialias=True),
                      T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])


class CropDataset(Dataset):
    def __init__(self, array_path, index, transform=None):
        self.x = np.load(array_path, mmap_mode='r')
        self.index = index.sort_values('row').reset_index(drop=True)
        self.y = self.index.category.map(CLASS_TO_ID).to_numpy()
        assert len(self.index) == len(self.x) and not np.isnan(self.y.astype(float)).any()
        self.transform = transform

    def __len__(self): return len(self.index)

    def __getitem__(self, i):
        img = np.array(self.x[i])          # copy out of the read-only memmap
        if self.transform is not None: img = self.transform(img)
        return img, int(self.y[i])


def class_weights(labels, power=0.5):
    """w_c proportional to 1 / n_c**power, mean weight over samples = 1. power 0 = none, 1 = full balance."""
    n = np.bincount(labels, minlength=len(CLASSES)).astype(float)
    w = np.where(n > 0, 1.0 / np.maximum(n, 1) ** power, 0.0)
    return w / (w[labels].mean())


def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); import random; random.seed(s)


def make_loader(ds, batch_size, train, seed=42, balance_power=0.5, num_workers=2):
    g = torch.Generator().manual_seed(seed)
    sampler = None
    if train and balance_power > 0:
        w = class_weights(ds.y, balance_power)[ds.y]
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(ds), replacement=True, generator=g)
    return DataLoader(ds, batch_size=batch_size, shuffle=train and sampler is None, sampler=sampler,
                      num_workers=num_workers, worker_init_fn=seed_worker, generator=g,
                      pin_memory=torch.cuda.is_available(), drop_last=train, persistent_workers=num_workers > 0)

## 1. Find the checkpoints and the test crops

In [ ]:
import os, re, sys, json, time, shutil, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
sys.path.insert(0, '/content'); import aerial_crops as ac; importlib.reload(ac)
pd.set_option('display.width', 200)
ROOT = Path(DRIVE_PROJECT); OUT_DIR = ROOT / '02_Classification'; CKPT_DIR = OUT_DIR / 'Checkpoints'
CROP_DIR = ROOT / '01_Data/Processed/Crops'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
K = len(ac.CLASSES); LABELS = list(range(K)); SEA = LABELS[:5]; URBAN = LABELS[5:]

TODAY = time.strftime('%Y-%m-%d')
_done = [int(m.group(1)) for f in os.listdir(OUT_DIR) if (m := re.match(rf'{TODAY}_TestResults_v(\d+)\.json$', f))]
VERSION = max(_done, default=0) + 1
def out_name(topic, ext): return f'{TODAY}_{topic}_v{VERSION}.{ext}'

def find_ckpt(run):
    hits = [(m.group(1), int(m.group(2)), f) for f in os.listdir(CKPT_DIR)
            if (m := re.match(rf'(\d{{4}}-\d{{2}}-\d{{2}})_{re.escape(run)}_v(\d+)\.pt$', f))]
    return CKPT_DIR / max(hits)[2] if hits else None

CKPTS = {}
for model_name, pat in EVAL.items():
    for s in SEEDS:
        p = find_ckpt(pat.format(seed=s))
        if p is None: print(f'missing: {pat.format(seed=s)} (skipped)'); continue
        CKPTS[(model_name, s)] = p
assert all((SELECTED, s) in CKPTS for s in SEEDS), f'the selected model needs all seeds {SEEDS}'
print(pd.Series({f'{m} s{s}': p.name for (m, s), p in CKPTS.items()}).to_string())

# all checkpoints must come from the same Step 4 crops
CK = {k: torch.load(p, map_location='cpu', weights_only=False) for k, p in CKPTS.items()}
used = {json.dumps(c['crops_used'], sort_keys=True) for c in CK.values()}
assert len(used) == 1, f'checkpoints trained on different crop files: {used}'
CROPS_USED = json.loads(used.pop()); print('crops:', CROPS_USED)
for k, c in CK.items(): assert c['classes'] == ac.CLASSES, k
assert all('logit_bias' in c for c in CK.values()), 'a checkpoint has no logit_bias: it was not finished by train_cls'

index = pd.read_csv(CROP_DIR / CROPS_USED['index'], keep_default_na=False)
TEST = index[index.split == 'test'].sort_values('row').reset_index(drop=True)
local = '/content/crops_test.npy'
if not os.path.exists(local) or os.path.getsize(local) != os.path.getsize(CROP_DIR / CROPS_USED['test']): shutil.copy(CROP_DIR / CROPS_USED['test'], local)
XT = np.load(local); assert len(XT) == len(TEST)
YT = TEST.category.map(ac.CLASS_TO_ID).to_numpy()
print('test crops:', len(TEST))

## 2. Models and inference function (same preprocessing as in training)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=8, width=32, dropout=0.3):
        super().__init__()
        layers, c_in = [], 3
        for i in range(4):
            c = width * 2 ** i
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

class Pretrained(nn.Module):
    def __init__(self, n_classes, mode, dropout):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)      # weights come from the checkpoint
        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(net.fc.in_features, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

def build(cfg):
    M = cfg['model']
    if M['name'] == 'SmallCNN': return SmallCNN(K, M['width'], M['dropout'])
    return Pretrained(K, M.get('mode'), M['dropout'])

MEAN = torch.tensor(ac.MEAN).view(1, 3, 1, 1); STD = torch.tensor(ac.STD).view(1, 3, 1, 1)
@torch.no_grad()
def logits_on(ck, X):
    m = build(ck['config']); m.load_state_dict(ck['model']); m.to(DEVICE).eval()
    img, gpu = ck['img_size'], ck['config']['data'].get('gpu_aug', False)
    tf = None if gpu else ac.make_transforms(img, train=False)
    out = []
    for i in range(0, len(X), 512):
        xb = X[i:i + 512]
        if gpu:        # train_cls gpu_eval: bilinear + antialias resize, normalise
            x = torch.from_numpy(xb).to(DEVICE).permute(0, 3, 1, 2).float() / 255
            if x.shape[-1] != img: x = F.interpolate(x, size=(img, img), mode='bilinear', antialias=True, align_corners=False)
            x = (x - MEAN.to(DEVICE)) / STD.to(DEVICE)
        else:
            x = torch.stack([tf(np.array(a)) for a in xb]).to(DEVICE)
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'): out.append(m(x).float().cpu())
    return torch.cat(out).numpy()


## 3. Pre-flight on val (test still unseen)

Re-scores every checkpoint on **val** with the inference code above and compares with the val macro-F1 recorded in training. If any run differs by more than 0.003, the notebook stops before test is opened.

In [ ]:
# Pre-flight on VAL (test still unseen): the same inference path must reproduce each checkpoint's val macro-F1.
# Catches a preprocessing or architecture mismatch BEFORE test is opened, and checks the index columns used later.
need = {'crop_id', 'dataset', 'category', 'source', 'track', 'image', 'x', 'y', 'w', 'h', 'box_side', 'size_bin', 'split', 'row'}
missing = need - set(index.columns); assert not missing, f'crop index lacks columns used later: {missing}'
EXPECTED = {  # val macro-F1 (raw, best epoch) from 02_Classification/2026-10-23_Step5/6/7Summary_v1 and _v2
    ('full FT', 42): 0.8165, ('full FT', 43): 0.8228, ('full FT', 44): 0.8113,            # lr 0.002 (Step7Summary_v3/v4)
    ('full FT lr0.001', 42): 0.8038, ('full FT lr0.001', 43): 0.8103, ('full FT lr0.001', 44): 0.7931,
    ('scratch', 42): 0.7783, ('scratch', 43): 0.7679, ('scratch', 44): 0.7906,
    ('frozen', 42): 0.7075, ('frozen', 43): 0.7074, ('frozen', 44): 0.7052}
VAL = index[index.split == 'val'].sort_values('row').reset_index(drop=True)
XV = np.load(CROP_DIR / CROPS_USED['val']); assert len(XV) == len(VAL)
YV = VAL.category.map(ac.CLASS_TO_ID).to_numpy()
rows = []
for k, ck in CK.items():
    pred = logits_on(ck, XV).argmax(1)
    f1 = float(precision_recall_fscore_support(YV, pred, labels=LABELS, zero_division=0)[2].mean())
    exp = EXPECTED.get(k)
    rows.append({'run': f'{k[0]} s{k[1]}', 'checkpoint': CKPTS[k].name, 'val_macro_f1_now': round(f1, 4),
                 'from_training': exp, 'diff': None if exp is None else round(f1 - exp, 4)})
pf = pd.DataFrame(rows); print(pf.to_string(index=False))
bad = pf[pf['diff'].abs() > 0.003]
assert bad.empty, 'val score NOT reproduced: inference differs from training. STOP here, do not open test; send this table.'
print('\nOK: val reproduced for every checkpoint (|diff| <= 0.003). Test can be opened.')
del XV


## 4. Test log (how often test is opened)

From here on test is open. **If a later cell fails, fix it and run from that cell. Do not use Run all again**: that appends a second entry to TestLog.csv.

In [ ]:
log_path = OUT_DIR / 'TestLog.csv'
entry = pd.DataFrame([{'when': time.strftime('%Y-%m-%d %H:%M'), 'notebook_version': VERSION,
                       'checkpoints': ';'.join(sorted(p.name for p in CKPTS.values()))}])
prev = pd.read_csv(log_path) if log_path.exists() else entry.iloc[:0]
if len(prev):
    print(f'test was opened {len(prev)} time(s) before:'); print(prev.to_string(index=False))
    if entry.checkpoints[0] not in set(prev.checkpoints):
        print('\nWARNING: these checkpoints differ from earlier test runs. If a model was changed after seeing test results, '
              'say so in the report (test was used more than once).')
pd.concat([prev, entry]).to_csv(log_path, index=False)

## 5. Test inference

In [ ]:
LOGITS = {}
for k, ck in CK.items():
    t0 = time.time(); LOGITS[k] = logits_on(ck, XT)
    print(f'{k[0]} s{k[1]}: {CKPTS[k].name}, {ck["img_size"]}px, gpu_aug {ck["config"]["data"].get("gpu_aug", False)}, {time.time() - t0:.0f}s')
BIAS = {k: np.array(ck['logit_bias']) for k, ck in CK.items()}


## 6. Metrics per run (raw and logit-adjusted), mean ± std over seeds

In [ ]:
def softmax(L): e = np.exp(L - L.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
def ece(P, Y, bins=15):
    conf, pred = P.max(1), P.argmax(1); e = 0.0; edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == Y[m]).mean() - conf[m].mean())
    return float(e)
def cm_fast(Y, pred): return np.bincount(Y * K + pred, minlength=K * K).reshape(K, K)
def f1_from_cm(cm):
    tp = np.diag(cm).astype(float); p = tp / np.maximum(cm.sum(0), 1); r = tp / np.maximum(cm.sum(1), 1)
    return np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0.0), p, r
def summary_metrics(Y, pred, present_only=False):
    """present_only (bootstrap draws): a class with no true crop in the draw is NaN, and macro averages skip it."""
    cm = cm_fast(Y, pred); f1, p, r = f1_from_cm(cm)
    if present_only:
        absent = cm.sum(1) == 0; f1, r = np.where(absent, np.nan, f1), np.where(absent, np.nan, r)
    mean = np.nanmean if present_only else np.mean
    return {'acc': float(np.trace(cm) / cm.sum()), 'macro_f1': float(mean(f1)),
            'macro_f1_sea': float(mean(f1[SEA])), 'macro_f1_urban': float(mean(f1[URBAN])),
            **{f'f1/{c}': float(v) for c, v in zip(ac.CLASSES, f1)}, **{f'recall/{c}': float(v) for c, v in zip(ac.CLASSES, r)},
            **{f'precision/{c}': float(v) for c, v in zip(ac.CLASSES, p)}}

PRED, ROWS = {}, []
for k, L in LOGITS.items():
    for kind, LL in (('raw', L), ('adjusted', L + BIAS[k])):
        pred = LL.argmax(1); PRED[(k[0], k[1], kind)] = pred
        ROWS.append({'model': k[0], 'seed': k[1], 'kind': kind, **summary_metrics(YT, pred), 'ece': ece(softmax(LL), YT)})
R = pd.DataFrame(ROWS)
main = ['acc', 'macro_f1', 'macro_f1_sea', 'macro_f1_urban', 'ece']
agg = R.groupby(['model', 'kind'])[main].agg(['mean', 'std']).round(4)
print('TEST, 64 px, argmax (raw) or argmax after the val-fitted logit bias (adjusted):'); print(agg.to_string())

## 7. Test composition: crops, groups and tracks per class

In [ ]:
comp = TEST.groupby('category').agg(crops=('crop_id', 'size'), groups=('source', 'nunique'),
                                    tracks=('track', lambda s: s[s != ''].nunique())).reindex(ac.CLASSES)
comp['dataset'] = ['SeaDronesSee'] * 5 + ['UAVDT'] * 3
print(comp.to_string())

## 8. 95 % CIs: group bootstrap (mean over seeds)

Each draw resamples SeaDronesSee videos and UAVDT sequences with replacement (separately, so the sea / urban mix stays), takes all their test crops, computes each metric for every seed and averages over seeds. The CI is the 2.5–97.5 percentile of 2,000 draws. jetski and LSA come from only 2 test videos each, so some draws contain no crop of them: those draws are left out for that class (column `draws_without_class`), and macro averages in such a draw use the classes present. Expect wide intervals for these classes; that is the honest result with 2 flights.

In [ ]:
grp = (TEST.dataset + '|' + TEST.source).to_numpy()
members = {g: np.flatnonzero(grp == g) for g in np.unique(grp)}
by_ds = {d: [g for g in members if g.startswith(d + '|')] for d in ('SeaDronesSee', 'UAVDT')}
rng = np.random.default_rng(SEED)
draws = [np.concatenate([members[g] for d in by_ds for g in rng.choice(by_ds[d], len(by_ds[d]), replace=True)]) for _ in range(N_BOOT)]
print({d: len(v) for d, v in by_ds.items()}, 'groups in test')

def boot(model, kind):
    preds = [PRED[(model, s, kind)] for s in SEEDS if (model, s, kind) in PRED]
    keys = None; out = []
    for idx in draws:
        y = YT[idx]; ms = [summary_metrics(y, p[idx], present_only=True) for p in preds]
        keys = keys or [k for k in ms[0] if not k.startswith('precision/')]
        out.append([np.mean([m[k] for m in ms]) for k in keys])
    A = np.array(out); full = [summary_metrics(YT, p, present_only=True) for p in preds]
    point = {k: np.mean([m[k] for m in full]) for k in keys}
    return pd.DataFrame({'mean_over_seeds': point, 'ci_low': dict(zip(keys, np.nanpercentile(A, 2.5, 0))),
                         'ci_high': dict(zip(keys, np.nanpercentile(A, 97.5, 0))),
                         'draws_without_class': dict(zip(keys, np.isnan(A).mean(0).round(3)))})

t0 = time.time()
CI = {(m, kind): boot(m, kind) for m in EVAL if any((m, s) in CKPTS for s in SEEDS) for kind in ('raw', 'adjusted')}
print(f'bootstrap done in {time.time() - t0:.0f}s')
sel = CI[(SELECTED, 'raw')].join(CI[(SELECTED, 'adjusted')], lsuffix=' raw', rsuffix=' adj').round(3)
print(f'\n{SELECTED}, test, mean of {len(SEEDS)} seeds, 95% group-bootstrap CI:'); print(sel.to_string())

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d
acc42 = (PRED[(SELECTED, SEEDS[0], 'raw')] == YT)
print(f'\nreference only, crop-level Wilson 95% CI for accuracy (seed {SEEDS[0]}, raw): '
      f'{acc42.mean():.4f} [{wilson(acc42.sum(), len(acc42))[0]:.4f}, {wilson(acc42.sum(), len(acc42))[1]:.4f}] '
      '(too narrow: crops of one video are correlated)')

## 9. Per-class table (selected model) with group counts

In [ ]:
pc = []
for c in ac.CLASSES:
    row = {'class': c, **comp.loc[c, ['crops', 'groups', 'tracks']].to_dict()}
    for kind in ('raw', 'adjusted'):
        sub = R[(R.model == SELECTED) & (R.kind == kind)]
        for met in ('precision', 'recall', 'f1'):
            row[f'{met} {kind}'] = f"{sub[f'{met}/{c}'].mean():.3f} ± {sub[f'{met}/{c}'].std():.3f}"
        ci = CI[(SELECTED, kind)]
        row[f'F1 CI {kind}'] = f"[{ci.loc[f'f1/{c}', 'ci_low']:.3f}, {ci.loc[f'f1/{c}', 'ci_high']:.3f}]"
    pc.append(row)
PC = pd.DataFrame(pc).set_index('class')
print(f'{SELECTED}, test, mean ± std over seeds {SEEDS}; CI = 95% group bootstrap of the seed mean:'); print(PC.to_string())
PC.to_csv(OUT_DIR / out_name('TestPerClass', 'csv'))

## 10. Confusion matrices (3 seeds pooled), printed and saved

In [ ]:
def show_cm(model, kind):
    cm = sum(confusion_matrix(YT, PRED[(model, s, kind)], labels=LABELS) for s in SEEDS if (model, s, kind) in PRED)
    df = pd.DataFrame(cm, index=pd.Index(ac.CLASSES, name='true'), columns=pd.Index(ac.CLASSES, name='predicted'))
    rowp = (df.div(df.sum(1).clip(lower=1), axis=0) * 100).round(1)
    print(f'\n{model} {kind}: counts (seeds pooled)'); print(df.to_string())
    print(f'{model} {kind}: row % (recall on the diagonal)'); print(rowp.to_string())
    df.to_csv(OUT_DIR / out_name(f'TestConfusion_{model.replace(" ", "")}_{kind}', 'csv'))
    n = rowp.to_numpy() / 100
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(n, cmap='Blues', vmin=0, vmax=1)
    for i in LABELS:
        for j in LABELS: ax.text(j, i, f'{rowp.iat[i, j]:.0f}', ha='center', va='center', fontsize=7, color='white' if n[i, j] > .5 else 'black')
    ax.set_xticks(LABELS, ac.CLASSES, rotation=45, ha='right', fontsize=8); ax.set_yticks(LABELS, ac.CLASSES, fontsize=8)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{model} ({kind}), test, 64 px, row %, {len(SEEDS)} seeds pooled', fontsize=9); plt.tight_layout()
    plt.savefig(OUT_DIR / out_name(f'TestConfusion_{model.replace(" ", "")}_{kind}', 'png'), dpi=130); plt.show()
    return df
CMS = {(m, kind): show_cm(m, kind) for m in [SELECTED] for kind in ('raw', 'adjusted')}

## 11. Calibration: reliability diagram (raw softmax, seeds pooled)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5)); edges = np.linspace(0, 1, 16)
for m in EVAL:
    ks = [(m, s) for s in SEEDS if (m, s) in LOGITS]
    if not ks: continue
    P = np.concatenate([softmax(LOGITS[k]) for k in ks]); Yp = np.tile(YT, len(ks))
    conf, ok = P.max(1), P.argmax(1) == Yp
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (conf > lo) & (conf <= hi)
        if b.sum() >= 20: xs.append(conf[b].mean()); ys.append(ok[b].mean())
    ax.plot(xs, ys, 'o-', ms=3, label=f'{m} (ECE {R[(R.model == m) & (R.kind == "raw")].ece.mean():.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.legend(fontsize=8)
ax.set_title('Test reliability, 64 px, 15 bins (bins < 20 crops hidden)', fontsize=9); plt.tight_layout()
plt.savefig(OUT_DIR / out_name('TestReliability', 'png'), dpi=130); plt.show()

## 12. Recall by object size (selected model, adjusted, seeds pooled)

In [ ]:
rows = []
for s in SEEDS:
    p = PRED[(SELECTED, s, 'adjusted')]
    rows.append(pd.DataFrame({'category': TEST.category, 'size_bin': TEST.size_bin, 'ok': p == YT}))
S = pd.concat(rows)
size_tab = S.pivot_table(index='category', columns='size_bin', values='ok', aggfunc='mean').reindex(ac.CLASSES)
size_n = (S.pivot_table(index='category', columns='size_bin', values='ok', aggfunc='size') / len(SEEDS)).reindex(ac.CLASSES)
order = [c for c in ['<16', '16-32', '32-64', '>64'] if c in size_tab]
print('recall by longer box side (px in the original image):'); print(size_tab[order].round(3).to_string())
print('crops per cell:'); print(size_n[order].fillna(0).astype(int).to_string())
size_tab[order].join(size_n[order].add_suffix(' n')).to_csv(OUT_DIR / out_name('TestRecallBySize', 'csv'))

## 13. Error gallery (selected model, seed 42, adjusted)

For the `GALLERY_PAIRS` most frequent (true → predicted) confusions: the `GALLERY_PER_PAIR` errors with the highest confidence (at most 2 per video or sequence), as stored crops (96 px, the model saw them at 64 px). Every error is also written to `…_TestErrors_vN.csv.gz` with its source image and box, for the data-improvement plan.

In [ ]:
s0 = SEEDS[0]; L0 = LOGITS[(SELECTED, s0)] + BIAS[(SELECTED, s0)]; P0 = softmax(L0); p0 = P0.argmax(1)
err = TEST.assign(pred=[ac.CLASSES[k] for k in p0], conf=P0.max(1).round(4))[p0 != YT]
err[['crop_id', 'dataset', 'category', 'pred', 'conf', 'image', 'x', 'y', 'w', 'h', 'box_side', 'source', 'track']] \
    .to_csv(OUT_DIR / out_name('TestErrors', 'csv.gz'), index=False)
pairs = err.groupby(['category', 'pred']).size().sort_values(ascending=False)
print(f'{len(err)} errors of {len(TEST)} ({len(err) / len(TEST):.1%}); top pairs:'); print(pairs.head(10).to_string())
top = pairs.head(GALLERY_PAIRS).index
fig, axes = plt.subplots(len(top), GALLERY_PER_PAIR, figsize=(GALLERY_PER_PAIR * 1.6, len(top) * 1.9), squeeze=False)
for i, (t, p) in enumerate(top):
    sub = err[(err.category == t) & (err.pred == p)].sort_values('conf', ascending=False)
    sub = sub.groupby('source', sort=False).head(2).head(GALLERY_PER_PAIR)      # at most 2 per video/sequence
    for j in range(GALLERY_PER_PAIR):
        ax = axes[i, j]; ax.axis('off')
        if j < len(sub):
            r = sub.iloc[j]; ax.imshow(XT[TEST.index[TEST.crop_id == r.crop_id][0]])
            ax.set_title(f'{r.conf:.2f} {int(r.box_side)}px\n{r.source[:12]}', fontsize=6)
    axes[i, 0].text(-0.15, 0.5, f'{t}\n-> {p}\n({pairs[(t, p)]})', transform=axes[i, 0].transAxes, ha='right', va='center', fontsize=8)
plt.suptitle(f'Test errors, {SELECTED} seed {s0}, adjusted; title: confidence, box size, source', fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / out_name('ErrorGallery', 'png'), dpi=120, bbox_inches='tight'); plt.show()
print('errors per source (top 10), to spot a single bad video or sequence:')
print(err.groupby(['dataset', 'source']).size().sort_values(ascending=False).head(10).to_string())

## 14. Save

In [ ]:
pred_out = TEST[['crop_id', 'dataset', 'category', 'source', 'box_side']].copy()
for (m, s, kind), p in PRED.items(): pred_out[f'{m.replace(" ", "")}_s{s}_{kind}'] = [ac.CLASSES[k] for k in p]
pred_out.to_csv(OUT_DIR / out_name('TestPredictions', 'csv.gz'), index=False)

def mean_std(model, kind):
    sub = R[(R.model == model) & (R.kind == kind)]
    return {k: {'mean': float(sub[k].mean()), 'std': float(sub[k].std()) if len(sub) > 1 else None, 'n_seeds': int(len(sub))}
            for k in main}
RESULT = {'created': time.strftime('%Y-%m-%d %H:%M'), 'version': VERSION, 'split': 'test', 'input_px': 64,
          'decision_rule': {'raw': 'argmax of softmax', 'adjusted': 'argmax after per-class logit bias fitted on val (Steps 5-7)'},
          'selected_model': SELECTED, 'seeds': SEEDS, 'checkpoints': {f'{m} s{s}': p.name for (m, s), p in CKPTS.items()},
          'crops_used': CROPS_USED, 'n_test_crops': int(len(TEST)), 'composition': comp.drop(columns='dataset').to_dict(orient='index'),
          'per_model': {m: {kind: mean_std(m, kind) for kind in ('raw', 'adjusted')} for m in R.model.unique()},
          'per_run': R.to_dict(orient='records'),
          'ci95_group_bootstrap': {f'{m} {kind}': ci.to_dict(orient='index') for (m, kind), ci in CI.items()},
          'ci_method': f'group bootstrap ({N_BOOT} draws, seed {SEED}) over SeaDronesSee videos and UAVDT sequences, resampled within dataset; metric = mean over seeds',
          'logit_bias': {f'{m} s{s}': BIAS[(m, s)].round(4).tolist() for (m, s) in BIAS},
          'n_errors_gallery_seed': int(len(err)), 'top_confusions': {f'{t}->{p}': int(n) for (t, p), n in pairs.head(10).items()},
          'test_log': str(log_path.name)}
out = OUT_DIR / out_name('TestResults', 'json')
json.dump(RESULT, open(out, 'w'), indent=2, default=float); print('saved', out)
print(json.dumps(RESULT['per_model'], indent=1)[:3000])